In [1]:
import pandas as pd
import numpy as np
import os
from glob import glob

# -----------------------------------------------
# 1. LOAD AND COMBINE ALL GW FILES
# -----------------------------------------------
# Finds every CSV in your GW folder and stacks them into one table

folder = r"C:\Users\JesseOnu\FPL new\gws 25_26"
all_files = glob(os.path.join(folder, "*.csv"))

df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

# -----------------------------------------------
# 2. FILTER TO PLAYERS WITH MINUTES > 0
# -----------------------------------------------
# Removes players who never played - stops them distorting percentiles

df = df[df["minutes"] > 0].copy()

# -----------------------------------------------
# 3. MINUTES SCORE (max-based, not percentile)
# -----------------------------------------------
# Max possible mins = highest gameweek number * 90
# Each player gets a % of that, scaled to 1-10

max_gw = df["gameweek"].max()
max_possible_mins = max_gw * 90

# Sum each player's total minutes across all GWs up to each GW
# We calculate cumulative minutes per player per GW
df = df.sort_values(["player_code", "gameweek"])
df["cumulative_mins"] = df.groupby("player_code")["minutes"].cumsum()

df["mins_score"] = (df["cumulative_mins"] / max_possible_mins * 9 + 1).clip(1, 10)

# -----------------------------------------------
# 4. PERCENTILE SCORES (within position, cumulative)
# -----------------------------------------------
# For each metric, we:
#   a) Calculate each player's cumulative total up to that GW
#   b) Rank them within their position at that GW snapshot
#   c) Convert rank to a 1-10 score

# Metrics where MORE = BETTER
positive_metrics = {
    "goals_scored": "goals_score",
    "assists":      "assists_score",
    "xG":           "xg_score",
    "xGI":          "xgi_score",
    "clean_sheets": "cs_score",
    "bonus":        "bonus_score",
}

# Metrics where LESS = BETTER (we invert the percentile)
negative_metrics = {
    "goals_conceded": "goals_conceded_score",
    "xGC":            "xgc_score",
}

def percentile_score(series, invert=False):
    """
    Takes a series of cumulative values.
    Returns a 1-10 score based on percentile rank.
    If invert=True, lower values get higher scores.
    """
    pct = series.rank(pct=True)   # gives each value a 0-1 percentile rank
    if invert:
        pct = 1 - pct             # flip so lower = better becomes higher score
    return (pct * 9 + 1).clip(1, 10)

# Calculate cumulative totals and percentile scores for each metric
for col, score_col in {**positive_metrics, **negative_metrics}.items():
    invert = col in negative_metrics
    
    # Cumulative total per player up to each GW
    cum_col = f"cum_{col}"
    df[cum_col] = df.groupby("player_code")[col].cumsum()
    
    # Percentile rank within position at each GW snapshot
    df[score_col] = (
        df.groupby(["gameweek", "position_id"])[cum_col]
        .transform(lambda x: percentile_score(x, invert=invert))
    )

# -----------------------------------------------
# 5. COMPOSITE SCORE (weighted average)
# -----------------------------------------------
# You can adjust these weights per position later
# For now, sensible defaults - same for all positions
# All weights should add up to 1

weights = {
    "mins_score":           0.1,
    "goals_score":          0.20,
    "assists_score":        0.10,
    "xg_score":             0.10,
    "xgi_score":            0.25,
    "cs_score":             0.10,
    "bonus_score":          0.05,
    "goals_conceded_score": 0.05,
    "xgc_score":            0.05,
}

df["composite_score"] = sum(
    df[score] * weight for score, weight in weights.items()
)

# -----------------------------------------------
# 6. SELECT OUTPUT COLUMNS
# -----------------------------------------------
# One row per player per GW with all scores

output_cols = [
    "player_code",
    "web_name",
    "position_id",
    "gameweek",
    "minutes",
    "mins_score",
    "goals_score",
    "assists_score",
    "xg_score",
    "xgi_score",
    "cs_score",
    "bonus_score",
    "goals_conceded_score",
    "xgc_score",
    "composite_score",
]

output = df[output_cols].sort_values(["gameweek", "composite_score"], ascending=[True, False])



In [2]:
output.head(15)

,player_code,web_name,position_id,gameweek,minutes,mins_score,goals_score,assists_score,xg_score,xgi_score,cs_score,bonus_score,goals_conceded_score,xgc_score,composite_score
429,223094,Haaland,4,1,72,1.257143,9.718750,5.500000,10.000000,10.000000,9.156250,9.578125,7.187500,5.781250,8.162433
6,466075,Calafiori,2,1,71,1.253571,9.955000,5.455000,10.000000,10.000000,8.425000,9.505000,7.750000,5.815000,8.157857
163,219249,O'Riley,3,1,87,1.310714,9.724490,5.010204,9.938776,9.969388,8.530612,9.908163,7.336735,6.387755,8.097908
530,223827,Ballard,2,1,90,1.321429,9.955000,5.455000,9.460000,8.920000,8.425000,9.865000,7.750000,7.570000,7.946393
426,433036,Reijnders,3,1,90,1.321429,9.724490,9.479592,8.806122,8.591837,8.530612,5.316327,7.336735,5.867347,7.832653
579,242898,Johnson,3,1,79,1.282143,9.724490,5.010204,9.755102,9.326531,8.530612,5.316327,7.336735,4.520408,7.593010
596,212319,Richarlison,4,1,71,1.253571,9.718750,5.500000,8.593750,8.593750,9.156250,9.578125,7.187500,3.812500,7.571451
660,510663,Ekitiké,4,1,71,1.253571,8.453125,10.000000,9.718750,9.718750,4.656250,8.734375,3.671875,2.968750,7.451920
346,243526,Gudmundsson,2,1,90,1.321429,5.455000,5.455000,9.235000,9.730000,8.425000,9.865000,7.750000,7.975000,7.246643
581,460842,Kudus,3,1,84,1.300000,5.255102,10.000000,6.877551,9.602041,8.530612,9.724490,7.336735,4.520408,7.201429


In [3]:
# -----------------------------------------------
# 7. SAVE OUTPUT
# -----------------------------------------------
output_path = r"C:\Users\JesseOnu\FPL new\fpl_player_scores.csv"
output.to_csv(output_path, index=False)

print(f"Done! {len(output)} rows saved to {output_path}")
print(output.head(10))

Done! 8500 rows saved to C:\Users\JesseOnu\FPL new\fpl_player_scores.csv
     player_code     web_name  position_id  gameweek  minutes  mins_score  \
429       223094      Haaland            4         1       72    1.257143   
6         466075    Calafiori            2         1       71    1.253571   
163       219249      O'Riley            3         1       87    1.310714   
530       223827      Ballard            2         1       90    1.321429   
426       433036    Reijnders            3         1       90    1.321429   
579       242898      Johnson            3         1       79    1.282143   
596       212319  Richarlison            4         1       71    1.253571   
660       510663      Ekitiké            4         1       71    1.253571   
346       243526  Gudmundsson            2         1       90    1.321429   
581       460842        Kudus            3         1       84    1.300000   

     goals_score  assists_score   xg_score  xgi_score  cs_score  bonus_score  \

In [5]:
df2 =  pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

In [6]:
df2.head()

,player_id,player_code,gameweek,fixture_id,name,web_name,position_id,selected_by_percent,value,team_id,...,ict_index,influence,creativity,threat,in_dreamteam,status,news,corners_indirect_freekicks_order,direct_freekicks_order,penalties_order
0,1,154561,1,9,David Raya Martín,Raya,1,34.4,6.0,1,...,4.9,49.2,0.0,0.0,True,a,NaN,NaN,NaN,NaN
1,2,109745,1,9,Kepa Arrizabalaga Revuelta,Arrizabalaga,1,0.4,4.1,1,...,0.0,0.0,0.0,0.0,False,a,NaN,NaN,NaN,NaN
2,3,463748,1,9,Karl Hein,Hein,1,0.2,4.0,1,...,0.0,0.0,0.0,0.0,False,u,Has joined Werder Bremen on loan for the rest ...,NaN,NaN,NaN
3,4,551221,1,9,Tommy Setford,Setford,1,0.2,3.9,1,...,0.0,0.0,0.0,0.0,False,a,NaN,NaN,NaN,NaN
4,5,226597,1,9,Gabriel dos Santos Magalhães,Gabriel,2,42.7,7.1,1,...,2.7,22.8,0.3,4.0,False,a,NaN,NaN,NaN,NaN
